# Tutorial 2: New Domain Modules

Explore comfio's new domain modules:

1. Pollutant IAQ (PM2.5, TVOC, formaldehyde, CO)
2. Simplified PMV (sPMV) — Buratti seasonal model
3. Adaptive Thermal Comfort (ASHRAE 55 & EN 16798)
4. TSV Augmentation (CDF-based remapping)
5. Personalised Comfort (OLS regression)
6. Integration with Global IEQ Index

In [ ]:
import numpy as np
from comfio import (
    evaluate_iaq_pollutants, evaluate_spmv,
    evaluate_adaptive_ashrae, evaluate_adaptive_en,
    augment_tsv_cdf, evaluate_tsv,
    train_personalisation, evaluate_personalised_pmv,
    calculate_global_ieq,
)

rng = np.random.default_rng(42)
n = 100

## 1. Pollutant IAQ

In [ ]:
pollutant = evaluate_iaq_pollutants(
    pm25=rng.normal(8.0, 2.0, n),
    pm10=rng.normal(15.0, 3.0, n),
    tvoc=rng.normal(150.0, 30.0, n),
    formaldehyde=rng.normal(20.0, 5.0, n),
    co=rng.normal(1.5, 0.5, n),
    threshold_level='good',
)
print(f'Pollutant IAQ score: {np.mean(pollutant.score):.1f}/100')
print(f'PM2.5 compliant: {np.mean(pollutant.compliant_pm25):.0%}')

## 2. Simplified PMV (sPMV)

In [ ]:
spmv = evaluate_spmv(
    indoor_temp=rng.normal(24.0, 1.5, n),
    indoor_rh=rng.normal(50.0, 5.0, n),
    season='mid',
)
print(f'Season: {spmv.season}')
print(f'Mean sPMV: {np.mean(spmv.spmv):.2f}')
print(f'Mean score: {np.mean(spmv.score):.1f}/100')

## 3. Adaptive Thermal Comfort

In [ ]:
tdb = rng.normal(24.0, 1.5, n)
tr = rng.normal(24.0, 1.0, n)

ashrae = evaluate_adaptive_ashrae(
    tdb=tdb, tr=tr, t_prevail=20.0, acceptability=80,
)
print(f'ASHRAE 55 — t_comf: {ashrae.t_comf:.1f}°C')
print(f'ASHRAE 55 — compliance: {np.mean(ashrae.compliant):.0%}')

en = evaluate_adaptive_en(
    tdb=tdb, tr=tr, t_running_mean=20.0, category='ii',
)
print(f'EN 16798 — t_comf: {en.t_comf:.1f}°C')
print(f'EN 16798 — compliance: {np.mean(en.compliant):.0%}')

## 4. TSV Augmentation

In [ ]:
sparse_votes = np.array([-2, -1, 0, 0, 1, 1, 2, -1, 0, 1], dtype=float)
augmented = augment_tsv_cdf(
    sparse_votes=sparse_votes,
    vote_timestamps=np.arange(10, dtype=float),
    target_timestamps=np.arange(n, dtype=float),
)
print(f'Source mean: {np.mean(sparse_votes):.2f}')
print(f'Augmented mean: {np.mean(augmented):.2f}')
print(f'Length: {len(augmented)}')

tsv_result = evaluate_tsv(augmented)
print(f'Compliance rate: {tsv_result.compliance_rate:.0%}')

## 5. Personalised Comfort

In [ ]:
pmv_hist = rng.normal(0.3, 0.5, n)
tsv_hist = np.clip(np.round(1.1 * pmv_hist + 0.2 + rng.normal(0, 0.3, n)), -3, 3)

index = train_personalisation(pmv=pmv_hist, tsv=tsv_hist)
print(f'Alpha: {index.alpha:.3f}, Beta: {index.beta:.3f}, R²: {index.r_squared:.3f}')

personalised = evaluate_personalised_pmv(
    tdb=tdb, tr=tr, vr=np.full(n, 0.1),
    rh=rng.normal(50.0, 5.0, n),
    met=1.2, clo=0.5,
    personalisation_index=index,
)
print(f'Personalised PMV (mean): {np.mean(personalised.personalised_pmv):.2f}')

## 6. Integration with Global IEQ

In [ ]:
from comfio import evaluate_thermal, evaluate_visual, evaluate_acoustic, evaluate_iaq

thermal = evaluate_thermal(
    tdb=tdb, tr=tr, vr=np.full(n, 0.1),
    rh=rng.normal(50.0, 5.0, n), met=1.2, clo=0.5,
)
visual = evaluate_visual(illuminance=rng.normal(500.0, 50.0, n))
acoustic = evaluate_acoustic(laeq=rng.normal(40.0, 5.0, n))
iaq = evaluate_iaq(co2=rng.normal(800.0, 100.0, n))

# Baseline (4 domains)
ieq_base = calculate_global_ieq(thermal=thermal, visual=visual, acoustic=acoustic, iaq=iaq)
print(f'Baseline IEQ: {np.mean(ieq_base.index):.1f}')

# With pollutant IAQ (blends 50/50 with CO₂)
ieq_poll = calculate_global_ieq(thermal=thermal, visual=visual, acoustic=acoustic, iaq=iaq, pollutant_iaq=pollutant)
print(f'With pollutant IAQ: {np.mean(ieq_poll.index):.1f}')

# With TSV (overrides thermal)
ieq_tsv = calculate_global_ieq(thermal=thermal, visual=visual, acoustic=acoustic, iaq=iaq, pollutant_iaq=pollutant, tsv=tsv_result)
print(f'With pollutant + TSV: {np.mean(ieq_tsv.index):.1f}')